# Load Models

In the previous notebook we saved the large model. In this notebook we have a look on how to reload it.

## Setup & Data

In [18]:
# Import packages
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import warnings

warnings.filterwarnings("ignore")

tf.keras.backend.set_floatx("float64")

# Define random seed for whole notebook
RSEED = 42

In [19]:
# Load data
df = pd.read_csv("../data/boston.csv")

# Define target
y = df.pop("MEDV")

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RSEED)

In [20]:
# Scale numerical features
# Scale numerical values
col_scale = [
    "CRIM",
    "ZN",
    "INDUS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "RAD",
    "TAX",
    "PTRATIO",
    "LSTAT",
]

scaler = StandardScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

# Convert to np array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

The SavedModel format is a directory containing a protobuf binary and a TensorFlow checkpoint. Inspect the saved model directory:

In [21]:
!ls saved_model

my_large_model.keras


In [22]:
!ls saved_model/my_large_model.keras

saved_model/my_large_model.keras


## SavedModel format

Reload a fresh Keras model from the saved model:

In [23]:
# Load the saved model
with tf.device("/cpu:0"):
    new_large_model = tf.keras.models.load_model("saved_model/my_large_model.keras")

# Check its architecture
new_large_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 512)            │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,385,412 (18.20 MB)

 Trainable params: 795,137 (6.07 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,590,275 (12.13 MB)

The restored model is compiled with the same arguments as the original model. Try running evaluate and predict with the loaded model:

In [24]:
# Evaluate the restored model
with tf.device("/cpu:0"):
    loss, mse = new_large_model.evaluate(X_test, y_test, verbose=2)
print(f"Model MSE: {mse}")

4/4 - 0s - 34ms/step - loss: 2.3157 - mse: 12.3281
Model MSE: 12.328079714525206


The Model MSE is higher than in the notebook before, even though we are using the same data-split. We are using the same model and the same data. Therefore, we would assume that we also receive the same MSE. 

> **Exercise:** Can you find out where this notebook varies from the procedure in the notebook before? It might help to have a look at the values in X_train in both notebooks.

<details><summary>
Click here for a hint...
</summary>
Check out how the data were preprocessed. Did both notebooks use the same scaler?
</details>

### Answer: why the reloaded model scored MSE 4,666 instead of 12.33
Two preprocessing differences from notebook 3, tested one at a time:

| Run | Scaler | RAD scaled | Test MSE |
|---|---|---|---|
| Original notebook 4 | MinMaxScaler | no | 4,666 |
| Fix scaler only | StandardScaler | no | 4,325 |
| Fix both | StandardScaler | yes | 12.33 (same as notebook 3) |

The model was fine; its input was not. Any mismatch, even one unscaled column, breaks it silently
(training-serving skew). The safe fix is to save the fitted scaler with joblib, or put the
scaling inside the model, instead of rebuilding preprocessing by hand.